In [1]:
pip install flask flask_sqlalchemy flask_login werkzeug

  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached flask_sqlalchemy-3.1.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached Flask_Login-0.6.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
Using cached flask-3.1.3-py3-none-any.whl (103 kB)
Using cached flask_sqlalchemy-3.1.1-py3-none-any.whl (25 kB)
Using cached Flask_Login-0.6.3-py3-none-any.whl (17 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)

   ---------------------------------------- 0/4 [jinja2]
   ---------------------------------------- 0/4 [jinja2]
   ---------------------------------------- 0/4 [jinja2]
   ---------------------------------------- 0/4 [jinja2]
   ---------- ----------------------------- 1/4 [flask]
   ---------- ----------------------------- 1/4 [flask]
   ---------- ----------------------------- 1/4 [flask]
   ---------- ----------------------------- 1/4 [flask]
   -------------------- ------------------- 2/4 [flask_sqlalc


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from flask import Flask, render_template, redirect, url_for, request, flash
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager, UserMixin, login_user, login_required, logout_user, current_user
from werkzeug.utils import secure_filename
from werkzeug.security import generate_password_hash, check_password_hash

app = Flask(__name__)
app.config['SECRET_KEY'] = 'your-secret-key'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///database.db'
app.config['UPLOAD_FOLDER'] = 'static/uploads'
app.config['ALLOWED_EXTENSIONS'] = {'png', 'jpg', 'jpeg'}

db = SQLAlchemy(app)
login_manager = LoginManager(app)
login_manager.login_view = 'login'

# Ensure upload folder exists
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

# --- Models ---

class User(UserMixin, db.Model):
    id = db.Column(db.Integer, primary_key=True)
    email = db.Column(db.String(100), unique=True)
    password = db.Column(db.String(100))
    display_name = db.Column(db.String(100))
    role = db.Column(db.String(20), default='viewer') # 'admin' or 'viewer'
    image_filename = db.Column(db.String(100), default='default.jpg')

class Student(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    fullname = db.Column(db.String(100))
    email = db.Column(db.String(100))

@login_manager.user_loader
def load_user(user_id):
    return User.query.get(int(user_id))

# --- Routes: Student Management ---

@app.route('/dashboard')
@login_required
def dashboard():
    students = Student.query.all()
    return render_template('dashboard.html', students=students)

@app.route('/add_student', methods=['POST'])
@login_required
def add_student():
    if current_user.role != 'admin':
        flash('Permission denied!')
        return redirect(url_for('dashboard'))
    
    name = request.form.get('fullname')
    email = request.form.get('email')
    new_student = Student(fullname=name, email=email)
    db.session.add(new_student)
    db.session.commit()
    flash('Student added successfully!')
    return redirect(url_for('dashboard'))

@app.route('/delete_student/<int:id>')
@login_required
def delete_student(id):
    if current_user.role != 'admin':
        flash('Permission denied!')
        return redirect(url_for('dashboard'))
    
    student = Student.query.get_or_404(id)
    db.session.delete(student)
    db.session.commit()
    flash('Student deleted.')
    return redirect(url_for('dashboard'))

In [4]:
def allowed_file(filename):
    return '.' in filename and \
           filename.rsplit('.', 1)[1].lower() in app.config['ALLOWED_EXTENSIONS']

@app.route('/profile', methods=['GET', 'POST'])
@login_required
def profile():
    if request.method == 'POST':
        name = request.form.get('display_name')
        file = request.files.get('profile_pic')

        # Update Display Name
        current_user.display_name = name

        # Handle Image Upload
        if file and allowed_file(file.filename):
            filename = secure_filename(f"user_{current_user.id}_{file.filename}")
            file.save(os.path.join(app.config['UPLOAD_FOLDER'], filename))
            current_user.image_filename = filename

        db.session.commit()
        flash('Profile updated successfully!')
        return redirect(url_for('dashboard'))
        
    return render_template('profile.html')

# Initialize DB
with app.app_context():
    db.create_all()